In [6]:
import pandas as pd
import os

RAW = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"

df_products  = pd.read_csv(os.path.join(RAW, 'products.csv'))
df_trends    = pd.read_csv(os.path.join(RAW, 'trend_signals.csv'))
df_inventory = pd.read_csv(os.path.join(RAW, 'inventory.csv'))
df_sales     = pd.read_csv(os.path.join(RAW, 'sales_transactions.csv'))
df_po        = pd.read_csv(os.path.join(RAW, 'purchase_orders.csv'))
df_social    = pd.read_csv(os.path.join(RAW, 'social_signals.csv'))
df_mapping   = pd.read_csv(os.path.join(RAW, 'trend_sku_mapping.csv'))

print("All dataframes reloaded ✅")
print(f"  products:          {len(df_products):>8,} rows")
print(f"  trend_signals:     {len(df_trends):>8,} rows")
print(f"  inventory:         {len(df_inventory):>8,} rows")
print(f"  sales_transactions:{len(df_sales):>8,} rows")
print(f"  purchase_orders:   {len(df_po):>8,} rows")
print(f"  social_signals:    {len(df_social):>8,} rows")
print(f"  trend_sku_mapping: {len(df_mapping):>8,} rows")

All dataframes reloaded ✅
  products:             1,170 rows
  trend_signals:        2,080 rows
  inventory:          122,850 rows
  sales_transactions: 120,000 rows
  purchase_orders:      8,000 rows
  social_signals:      37,440 rows
  trend_sku_mapping:    1,200 rows


In [7]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import pandas as pd
import os
import time

# ── Database Connection ────────────────────────────────────────
DB_USER     = "postgres"
DB_PASSWORD = quote_plus(os.environ["DB_PASSWORD"])
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "fashion_trend_db"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ── Test Connection ────────────────────────────────────────────
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected to:", result.fetchone()[0])
    print("Connection successful ✅\n")

# ── Load All Tables ────────────────────────────────────────────
tables = {
    'products':           df_products,
    'trend_signals':      df_trends,
    'inventory':          df_inventory,
    'sales_transactions': df_sales,
    'purchase_orders':    df_po,
    'social_signals':     df_social,
    'trend_sku_mapping':  df_mapping
}

print("=" * 55)
print("  LOADING DATA INTO POSTGRESQL")
print("=" * 55)

total_start = time.time()

for table_name, df in tables.items():
    start = time.time()
    df.to_sql(
        name        = table_name,
        con         = engine,
        if_exists   = 'append',
        index       = False,
        method      = 'multi',
        chunksize   = 1000
    )
    elapsed = round(time.time() - start, 1)
    print(f"  ✅ {table_name:<25} {len(df):>8,} rows  ({elapsed}s)")

total_elapsed = round(time.time() - total_start, 1)
print("-" * 55)
print(f"  Total time: {total_elapsed}s")
print("=" * 55)
print("\nAll tables loaded into fashion_trend_db ✅")

Connected to: PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35213, 64-bit
Connection successful ✅

  LOADING DATA INTO POSTGRESQL
  ✅ products                     1,170 rows  (0.8s)
  ✅ trend_signals                2,080 rows  (1.2s)
  ✅ inventory                  122,850 rows  (79.4s)
  ✅ sales_transactions         120,000 rows  (116.2s)
  ✅ purchase_orders              8,000 rows  (5.3s)
  ✅ social_signals              37,440 rows  (20.6s)
  ✅ trend_sku_mapping            1,200 rows  (0.4s)
-------------------------------------------------------
  Total time: 223.9s

All tables loaded into fashion_trend_db ✅


In [ ]:
# ── Fix: Inject Realistic Stockouts into Inventory ────────────
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

RAW = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"

# Reload inventory
df_inventory = pd.read_csv(os.path.join(RAW, 'inventory.csv'))

np.random.seed(42)

# Select 15% of SKUs to experience stockout periods
all_skus = df_inventory['sku_id'].unique()
stockout_skus = np.random.choice(all_skus, size=int(len(all_skus)*0.15), replace=False)

stockout_count = 0

for sku in stockout_skus:
    sku_mask = df_inventory['sku_id'] == sku
    sku_indices = df_inventory[sku_mask].index.tolist()

    # Pick 1-3 random stockout windows per SKU
    num_windows = np.random.randint(1, 4)

    for _ in range(num_windows):
        # Random start point (not first or last 10 weeks)
        start_idx = np.random.randint(10, len(sku_indices) - 15)
        duration  = np.random.randint(2, 6)  # 2-5 weeks of stockout

        for w in range(duration):
            idx = sku_indices[start_idx + w]
            df_inventory.at[idx, 'closing_stock']   = 0
            df_inventory.at[idx, 'opening_stock']   = 0 if w > 0 else df_inventory.at[idx, 'opening_stock']
            df_inventory.at[idx, 'units_sold']      = 0
            df_inventory.at[idx, 'stockout_flag']   = 1
            df_inventory.at[idx, 'inventory_health']= 'Critical'
            df_inventory.at[idx, 'weeks_of_cover']  = 0.0
            df_inventory.at[idx, 'inventory_value'] = 0.0
            stockout_count += 1

# Save updated inventory
df_inventory.to_csv(os.path.join(RAW, 'inventory.csv'), index=False)

print(f"Stockout records injected: {stockout_count:,}")
print(f"SKUs with stockouts:       {len(stockout_skus):,}")
print(f"Total stockout_flag = 1:   {df_inventory['stockout_flag'].sum():,}")
print("Inventory CSV updated ✅")